# COMPARATIVE ANALYSIS OF NEURON MODELS

Models implemented
------------------
1. Leaky Integrate and Fire
2. Izhikevich
3. Hodgkin-Huxley

Analysis performed
------------------
- Voltage dynamics
- Spike detection
- Firing rate
- Inter spike interval
- F-I curves

## Import Libraries

In [3]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time

## Input Current Function

In [4]:
def input_current(t):
    """
    External stimulus current.

    Parameters
    ----------
    t : float
        time in ms

    Returns
    -------
    float
        injected current
    """

    if 50 <= t <= 300:
        return 10
    return 0


## Spike Detection Utility

In [5]:
def detect_spikes(voltage, threshold=0):
    """
    Detect spike times from voltage trace.

    Parameters
    ----------
    voltage : array
        membrane voltage
    threshold : float
        spike detection threshold

    Returns
    -------
    list
        spike indices
    """

    spikes = []

    for i in range(1, len(voltage)):

        if voltage[i-1] < threshold and voltage[i] >= threshold:
            spikes.append(i)

    return spikes

## LIF MODEL

Initialize V = V_rest

For each timestep:

    compute input current I(t)

    update membrane voltage

    dV/dt = (-(V - V_rest) + R*I)/tau

    V = V + dt*dV/dt

    if V >= threshold:
        record spike
        V = reset

In [6]:
class LIFNeuron:

    """
    Implementation of Leaky Integrate and Fire neuron.
    """

    def __init__(self, T=400, dt=0.1):

        self.dt = dt
        self.time = np.arange(0, T, dt)

        self.V_rest = -65
        self.V_reset = -75
        self.threshold = -50
        self.tau = 20

        self.V = np.zeros(len(self.time))
        self.V[0] = self.V_rest


    def simulate(self):

        for i in range(1, len(self.time)):

            V_prev = self.V[i-1]
            I = input_current(self.time[i])

            dV = (-(V_prev - self.V_rest) + I)/self.tau
            V = V_prev + self.dt*dV

            if V >= self.threshold:
                self.V[i] = 30
                if i+1 < len(self.V):
                    self.V[i+1] = self.V_reset
            else:
                self.V[i] = V

        return self.V

## IZHIKEVICH MODEL

Initialize V, u

For each timestep:

    dV/dt = 0.04V² + 5V + 140 − u + I
    du/dt = a(bV − u)

    update variables

    if V ≥ 30:
        spike
        V = c
        u = u + d

In [7]:
class IzhikevichNeuron:

    """
    Izhikevich neuron capable of reproducing many firing patterns.
    """

    def __init__(self, T=400, dt=0.1):

        self.dt = dt
        self.time = np.arange(0, T, dt)

        self.a = 0.02
        self.b = 0.2
        self.c = -65
        self.d = 8

        self.V = np.zeros(len(self.time))
        self.u = np.zeros(len(self.time))

        self.V[0] = -65
        self.u[0] = self.b*self.V[0]


    def simulate(self):

        for i in range(1, len(self.time)):

            V = self.V[i-1]
            u = self.u[i-1]

            I = input_current(self.time[i])

            dV = 0.04*V*V + 5*V + 140 - u + I
            du = self.a*(self.b*V - u)

            V += self.dt*dV
            u += self.dt*du

            if V >= 30:
                self.V[i] = 30
                V = self.c
                u += self.d
            else:
                self.V[i] = V

            self.u[i] = u

        return self.V

## HODGKIN HUXLEY MODEL

Compute gating variables m,h,n

Update channel currents:

INa = gNa*m³h*(V − ENa)
IK = gK*n⁴*(V − EK)
IL = gL*(V − EL)

Update membrane voltage:

dV/dt = (I − INa − IK − IL)/Cm

In [8]:
class HodgkinHuxleyNeuron:

    """
    Full Hodgkin-Huxley neuron model.
    """

    def __init__(self, T=400, dt=0.01):

        self.dt = dt
        self.time = np.arange(0, T, dt)

        self.Cm = 1
        self.gNa = 120
        self.gK = 36
        self.gL = 0.3

        self.ENa = 50
        self.EK = -77
        self.EL = -54.4

        self.V = np.zeros(len(self.time))
        self.V[0] = -65

        self.m = np.zeros(len(self.time))
        self.h = np.zeros(len(self.time))
        self.n = np.zeros(len(self.time))

        self.m[0] = 0.05
        self.h[0] = 0.6
        self.n[0] = 0.32


    def alpha_m(self,V):
        return 0.1*(V+40)/(1-np.exp(-(V+40)/10))

    def beta_m(self,V):
        return 4*np.exp(-(V+65)/18)

    def alpha_h(self,V):
        return 0.07*np.exp(-(V+65)/20)

    def beta_h(self,V):
        return 1/(1+np.exp(-(V+35)/10))

    def alpha_n(self,V):
        return 0.01*(V+55)/(1-np.exp(-(V+55)/10))

    def beta_n(self,V):
        return 0.125*np.exp(-(V+65)/80)


    def simulate(self):

        for i in range(1,len(self.time)):

            V = self.V[i-1]

            I = input_current(self.time[i])

            m = self.m[i-1]
            h = self.h[i-1]
            n = self.n[i-1]

            dm = self.alpha_m(V)*(1-m) - self.beta_m(V)*m
            dh = self.alpha_h(V)*(1-h) - self.beta_h(V)*h
            dn = self.alpha_n(V)*(1-n) - self.beta_n(V)*n

            m += self.dt*dm
            h += self.dt*dh
            n += self.dt*dn

            INa = self.gNa*m**3*h*(V-self.ENa)
            IK = self.gK*n**4*(V-self.EK)
            IL = self.gL*(V-self.EL)

            dV = (I-INa-IK-IL)/self.Cm
            V += self.dt*dV

            self.V[i]=V
            self.m[i]=m
            self.h[i]=h
            self.n[i]=n

        return self.V

## VISUALIZATION

In [9]:
def plot_voltage(time, lif, izh, hh):

    fig = go.Figure()

    fig.add_trace(go.Scatter(x=time,y=lif,name="LIF"))
    fig.add_trace(go.Scatter(x=time,y=izh,name="Izhikevich"))
    fig.add_trace(go.Scatter(x=time,y=hh[:len(time)],name="Hodgkin-Huxley"))

    fig.update_layout(
        title="Neuron Model Voltage Comparison",
        xaxis_title="Time (ms)",
        yaxis_title="Voltage (mV)",
        template="plotly_white"
    )

    fig.show()

## F-I CURVE

In [10]:
def FI_curve(model_class):

    currents = np.linspace(2,20,15)
    firing_rates=[]

    for I in currents:

        def current(t):
            return I

        neuron=model_class()

        global input_current
        input_current=current

        V=neuron.simulate()

        spikes=detect_spikes(V)

        rate=len(spikes)/0.4

        firing_rates.append(rate)

    return currents,firing_rates

## MAIN FUNCTION

In [11]:
def main():

    start=time.time()

    lif=LIFNeuron()
    izh=IzhikevichNeuron()
    hh=HodgkinHuxleyNeuron()

    lif_v=lif.simulate()
    izh_v=izh.simulate()
    hh_v=hh.simulate()

    runtime=time.time()-start

    print("Simulation time:",runtime)

    plot_voltage(lif.time,lif_v,izh_v,hh_v)


if __name__=="__main__":
    main()

Simulation time: 0.6534256935119629
